In [ ]:
#Week 6, Day 4 — June 25, 2026
#Import Final Data Layers into Power BI

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings, datetime
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/HCL_Project"
DIRS = {"raw":BASE_DIR+"/data/raw","processed":BASE_DIR+"/data/processed",
        "metadata":BASE_DIR+"/data/metadata","visualizations":BASE_DIR+"/visualizations"}
print("Environment ready ")

In [ ]:
# Load all 4 dashboard datasets
master      = pd.read_csv(DIRS['processed']+'/master_table_v2.csv')
predictions = pd.read_csv(DIRS['processed']+'/dashboard_ready_data.csv')
risk_zones  = pd.read_csv(DIRS['processed']+'/risk_zone_summary.csv')
hotspots    = pd.read_csv(DIRS['processed']+'/hotspots.csv')

print('DASHBOARD DATASETS LOADED')
print('='*55)
print(f'  master_table_v2.csv      : {master.shape[0]:,} rows × {master.shape[1]} cols')
print(f'  dashboard_ready_data.csv : {predictions.shape[0]:,} rows × {predictions.shape[1]} cols')
print(f'  risk_zone_summary.csv    : {risk_zones.shape[0]:,} rows × {risk_zones.shape[1]} cols')
print(f'  hotspots.csv             : {hotspots.shape[0]:,} rows × {hotspots.shape[1]} cols')

In [ ]:
#Step 1: Inspect Each Dataset — Schema Review
print('SCHEMA REVIEW — MASTER TABLE (Fact Table)')
print('='*55)
print(f'  Primary keys  : grid_lat, grid_lon, year, month')
print(f'  Dimension keys: Ocean_Zone, Country')
print(f'  Metric cols   : Plastic_Density_kg_km2, Species_Count, SST_C')
print(f'  KPI cols      : Temp_Change_C, Regional_Contribution_Pct')
print()
print('  Columns:')
for col in master.columns:
    dtype   = master[col].dtype
    n_null  = master[col].isnull().sum()
    n_uniq  = master[col].nunique()
    print(f'    {col:<35} {str(dtype):<12} nulls={n_null:<5} unique={n_uniq}')

In [ ]:
print('SCHEMA REVIEW — PREDICTION TABLE')
print('='*55)
print(f'  Primary keys  : Date, Location, Risk_Cluster')
print(f'  Metric cols   : SST, pH, Species Observed, RF_Predicted_Species')
print()
for col in predictions.columns:
    dtype  = predictions[col].dtype
    n_null = predictions[col].isnull().sum()
    n_uniq = predictions[col].nunique()
    print(f'    {col:<35} {str(dtype):<12} nulls={n_null:<5} unique={n_uniq}')
print()
print('SCHEMA REVIEW — RISK ZONE SUMMARY (Dimension)')
print('='*55)
print(risk_zones.to_string(index=False))

In [ ]:
## Step 2: Define Data Model Relationships (Star Schema)
print('POWER BI / TABLEAU STAR SCHEMA')
print('='*65)
print()
print('                  ┌─────────────────────┐')
print('                  │   risk_zone_summary  │  (dimension)')
print('                  │   Risk_Zone (PK)     │')
print('                  └────────┬────────────┘')
print('                           │ Many-to-One')
print('          ┌────────────────┼───────────────────┐')
print('          │                │                   │')
print('   ┌──────▼──────┐  ┌──────▼──────┐   ┌───────▼──────┐')
print('   │master_table │  │ predictions  │   │  hotspots    │')
print('   │(fact table) │  │(pred table)  │   │ (dimension)  │')
print('   │Ocean_Zone   │  │Risk_Cluster  │   │grid_lat/lon  │')
print('   │grid_lat/lon │  │Date,Location │   │              │')
print('   └─────────────┘  └─────────────┘   └──────────────┘')
print()

relationships = [
    ('master_table_v2',     'Ocean_Zone',   'risk_zone_summary',  'Risk_Zone',    'Many-to-One'),
    ('dashboard_ready_data','Risk_Cluster', 'risk_zone_summary',  'Risk_Zone',    'Many-to-One'),
    ('master_table_v2',     'grid_lat/lon', 'hotspots',           'grid_lat/lon', 'Many-to-One'),
]
print('RELATIONSHIP DEFINITIONS:')
print(f'  {"Fact Table":<28} {"FK":<18} {"Dimension":<22} {"PK":<14} Type')
print('-'*95)
for fact,fk,dim,pk,rel in relationships:
    print(f'  {fact:<28} [{fk:<16}] → {dim:<22} [{pk:<12}] {rel}')
print()
print('NOTE: dashboard_ready_data and master_table_v2 can be linked via')
print('      year/month keys for combined temporal analysis in the dashboard.')

In [ ]:
## Step 3: Verify All Filter Slicer Fields
print('DASHBOARD FILTER SLICER VERIFICATION')
print('='*55)
print()

slicer_checks = [
    ('Country',              master,      'India, global context'),
    ('Waste_Source',         master,      'Consumer_Goods, Industrial'),
    ('Ocean_Zone',           master,      'Arabian_Sea, Bay_of_Bengal, etc.'),
    ('Risk_Cluster',         predictions, 'Stable, At_Risk, Critical'),
    ('year',                 master,      '2015 (plastic), 2015-2026 (species)'),
    ('Plastic_Types_Present',master,      'PET, PE, PS, PP, PVC'),
]

for field, df_check, description in slicer_checks:
    present = field in df_check.columns
    status  = '✅' if present else '❌ MISSING'
    if present:
        vals = sorted(df_check[field].dropna().unique())[:4]
        print(f'  {status} {field:<28} sample: {vals}')
    else:
        print(f'  {status} {field}')
    print(f'      → Used for: {description}')
    print()

In [ ]:
#Step 4: Data Type Alignment for Power BI
print('DATA TYPE ALIGNMENT — POWER BI REQUIREMENTS')
print('='*55)
print()

# Fix Date column in predictions (ensure string → datetime in Power BI)
predictions_fixed = predictions.copy()
if predictions_fixed['Date'].dtype == object:
    predictions_fixed['Date'] = pd.to_datetime(predictions_fixed['Date'])
    print('  Date column: converted to datetime ')

# Ensure numeric cols are correct
numeric_check = {
    'master':      ['Plastic_Density_kg_km2','Species_Count','SST_C','year','month'],
    'predictions': ['SST (°C)','pH Level','Species Observed','RF_Predicted_Species','Latitude','Longitude'],
}
for table_name, cols in numeric_check.items():
    df_check = master if table_name == 'master' else predictions
    print(f'  {table_name} — numeric columns:')
    for col in cols:
        if col in df_check.columns:
            dtype = df_check[col].dtype
            is_num = pd.api.types.is_numeric_dtype(df_check[col])
            status = '✅' if is_num else '⚠️  needs cast'
            print(f'    {col:<35} {str(dtype):<12} {status}')
    print()

print('All dtypes confirmed for Power BI import ')

In [ ]:
#Step 5: Save Power BI Schema Documentation
schema_doc = DIRS['metadata']+'/powerbi_data_model.txt'

with open(schema_doc,'w') as f:
    f.write('POWER BI / TABLEAU DATA MODEL DOCUMENTATION\n')
    f.write(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write('='*65+'\n\n')

    f.write('STAR SCHEMA OVERVIEW:\n')
    f.write('  FACT TABLE        : master_table_v2.csv\n')
    f.write('  PREDICTION TABLE  : dashboard_ready_data.csv\n')
    f.write('  DIMENSION TABLE 1 : risk_zone_summary.csv\n')
    f.write('  DIMENSION TABLE 2 : hotspots.csv\n\n')

    f.write('RELATIONSHIPS:\n')
    for fact,fk,dim,pk,rel in relationships:
        f.write(f'  {fact}[{fk}] → {dim}[{pk}]  ({rel})\n')
    f.write('\n')

    f.write('FACT TABLE — master_table_v2.csv:\n')
    for col in master.columns:
        f.write(f'  {col}: {master[col].dtype}\n')

    f.write('\nPREDICTION TABLE — dashboard_ready_data.csv:\n')
    for col in predictions.columns:
        f.write(f'  {col}: {predictions[col].dtype}\n')

    f.write('\nDIMENSION — risk_zone_summary.csv:\n')
    for col in risk_zones.columns:
        f.write(f'  {col}: {risk_zones[col].dtype}\n')

    f.write('\nFILTER SLICERS (5 total):\n')
    slicers = [
        ('Country',       'master_table_v2',     'India'),
        ('Waste_Source',  'master_table_v2',     'Consumer_Goods, Industrial'),
        ('Ocean_Zone',    'master_table_v2',     'Arabian_Sea, Bay_of_Bengal, Andaman_Sea, Indian_Ocean, Gulf_of_Kutch'),
        ('Risk_Cluster',  'dashboard_ready_data','Stable, At_Risk, Critical'),
        ('year',          'master_table_v2',     '2015-2026'),
    ]
    for slicer,source,vals in slicers:
        f.write(f'  {slicer} (source: {source}): {vals}\n')

    f.write('\nSTEPS IN POWER BI DESKTOP:\n')
    steps = [
        '1. Get Data → Text/CSV → load all 4 CSV files from Google Drive',
        '2. Transform Data → verify dtypes → set Date col as Date type',
        '3. Model View → draw relationships as documented above',
        '4. Report View → Add visuals: KPI cards, pie chart, map, bar chart',
        '5. Add 5 Slicers: Country, Waste_Source, Ocean_Zone, Risk_Cluster, year',
        '6. Connect Folium HTML map as embedded visual or link',
        '7. File → Save As → .pbix → export to Google Drive',
    ]
    for step in steps:
        f.write(f'  {step}\n')

print(f'Power BI schema documentation saved ')
with open(schema_doc) as f: print(f.read())